# 07 规则与策略分析

每节下方的完整代码块均可单独复制运行，包含自己的 import、数据加载和实际调用。使用全部订单，不做行采样。安装所需依赖后，从 `examples` 目录运行。

In [ ]:
import sys
sys.path.append('../')

## 1. 规则运算、命中与过滤

直接定义规则并展示与、或、非、异或；数据使用全部订单。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from hscredit import Rule

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

high_query = Rule("BASE_X187 >= 60", name="多头查询偏多", n_jobs=1)
high_accounts = Rule("DK_X1689 >= 12", name="现行账户偏多", n_jobs=1)
industry = Rule("BASE_X61 in ['建筑业', '制造业']", name="指定行业", n_jobs=1)
degree = Rule("BASINF_X19 in ['硕士', '博士']", name="指定学位", n_jobs=1)

both = high_query & high_accounts
either = high_query | high_accounts
inverse = ~degree
exclusive = high_query ^ high_accounts
combined = both | (industry & inverse)

hits = pd.DataFrame(
    {
        "与": both.predict(data),
        "或": either.predict(data),
        "非": inverse.predict(data),
        "异或": exclusive.predict(data),
        "组合规则": combined.predict(data),
    }
)
display(hits.head())
display(hits.mean().rename("命中率"))
display(combined.filter(data).head())

,与,或,非,异或,组合规则
0,False,True,True,True,True
1,True,True,True,False,True
2,False,True,True,True,False
3,False,True,True,True,True
4,True,True,True,False,True


与      0.3493
或      0.9440
非      0.9881
异或     0.5947
组合规则   0.4345
Name: 命中率, dtype: float64

,用户编码,授信时间,MOB1,MOB6,BASE_X187,loan_amt,GROUP,BASE_X257,BASE_X304,DK_X1995_2,...,DK_X2536,DK_X1560,DKD1_X144,DKD1_X575,DK_X2482,DJK_X614,DJK_X470,DK_X628,BASINF_X19,BASE_X61
0,USER100001,2024-02-07 14:20:14,0.0000,487.0000,26.0000,5000.0000,衍生客群,12.0000,12.0000,142700.0000,...,12.0000,4924.0000,NaN,1.0000,0.0000,0.0000,0.0000,7417.0000,学士,制造业
1,USER100002,2024-02-09 18:40:26,0.0000,0.0000,76.0000,5000.0000,优质客群-消费金融类,31.0000,29.0000,376136.0000,...,22.0000,NaN,NaN,0.0000,0.0000,0.0000,0.0000,3309.0000,--,--
3,USER100004,2024-02-27 16:32:19,1.0000,1.0000,38.0000,3000.0000,高危客群,14.0000,21.0000,349218.0000,...,16.0000,3483.0000,NaN,1.0000,0.0000,0.0000,0.0000,851.0000,无,建筑业
4,USER100005,2024-03-10 18:25:33,0.0000,0.0000,71.0000,5000.0000,优质客群-消费金融类,21.0000,22.0000,574350.0000,...,11.0000,NaN,NaN,0.0000,0.0000,0.0000,1.0000,8563.0000,无,公共管理、社会保障和社会组织
8,USER100009,2024-04-02 16:03:31,0.0000,0.0000,60.0000,5000.0000,优质客群-消费金融类,18.0000,27.0000,399229.0000,...,2.0000,NaN,NaN,0.0000,0.0000,0.0000,0.0000,6475.0000,--,--


## 2. 规则 report 与多规则分析

同一规则同时展示多期逾期标签，以及订单和金额口径。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from hscredit import Rule
from hscredit.report import ruleset_analysis

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

high_query = Rule("BASE_X187 >= 60", name="多头查询偏多", n_jobs=1)
high_accounts = Rule("DK_X1689 >= 12", name="现行账户偏多", n_jobs=1)
industry = Rule("BASE_X61 in ['建筑业', '制造业']", name="指定行业", n_jobs=1)
degree = Rule("BASINF_X19 in ['硕士', '博士']", name="指定学位", n_jobs=1)

combined = high_query & high_accounts
display(
    combined.report(
        data, overdue=["MOB1", "MOB3", "MOB6"], dpds=30, overdue_operator=">", amount="loan_amt"
    )
)

analysis = ruleset_analysis(
    data,
    rules=[high_query, high_accounts, industry, degree],
    overdue=["MOB1", "MOB3", "MOB6"],
    dpds=[0, 30],
    overdue_operator=">",
    amount="loan_amt",
    n_jobs=1,
)
display(analysis)

分箱详情                                                                    MOB1 30+                                    ...      MOB6 30+                                                                 
   规则分类                              指标名称   分箱           样本总数   样本占比           好样本数  好样本占比         坏样本数  坏样本占比   坏样本率  ...          坏样本数  坏样本占比   坏样本率  LIFT值    坏账改善   风险拒绝比    准确率    精确率    召回率   F1分数
0  验证规则  BASE_X187 >= 60 & DK_X1689 >= 12   命中 288514000.0000 0.3862 286606000.0000 0.3882 1908000.0000 0.2168 0.0066  ... 19805000.0000 0.3000 0.0686 0.7766 -0.1405 -0.3639 0.6089 0.0697 0.2688 0.1107
1  验证规则  BASE_X187 >= 60 & DK_X1689 >= 12  未命中 458515000.0000 0.6138 451621000.0000 0.6118 6894000.0000 0.7832 0.0150  ... 46222000.0000 0.7000 0.1008 1.1405  0.2234  0.3639 0.3911 0.1018 0.7312 0.1787

[2 rows x 41 columns]

分箱详情                              MOB1 0+                                                    ...      MOB6 30+                                                                 
                           分箱           样本总数   样本占比           好样本数  好样本占比          坏样本数  坏样本占比   坏样本率  LIFT值    坏账改善  ...          坏样本数  坏样本占比   坏样本率  LIFT值    坏账改善   风险拒绝比    准确率    精确率    召回率   F1分数
0                        原始样本 747029000.0000 1.0000 677204000.0000 1.0000 69825000.0000 1.0000 0.0935 1.0000  0.0000  ... 66027000.0000 1.0000 0.0884 1.0000  0.0000  0.0000 0.1279 0.0892 0.9364 0.1628
1             BASE_X187 >= 60 289441000.0000 0.3875 272717000.0000 0.4027 16724000.0000 0.2395 0.0578 0.6182 -0.2415  ... 19871000.0000 0.3010 0.0687 0.7767 -0.1412 -0.3645 0.6075 0.0697 0.2699 0.1108
2                        剩余样本 457588000.0000 0.6125 404487000.0000 0.5973 53101000.0000 0.7605 0.1160 1.2415  0.3818  ... 46156000.0000 0.6990 0.1009 1.1412  0.2233  0.3645 0.3925 0.1019 0.7301 0.1788
3              DK_X1689 >= 12 425179000.0000 0.9292 379206000.0000 0.9375 45973000.0000 0.8658 0.1081 0.9318 -0.8953  ... 42159000.0000 0.9134 0.0992 0.9830 -0.2227 -0.2397 0.1673 0.1001 0.8981 0.1801
4                        剩余样本  32409000.0000 0.0708  25281000.0000 0.0625  7128000.0000 0.1342 0.2199 1.8953  0.0682  ...  3997000.0000 0.0866 0.1233 1.2227  0.0170  0.2397 0.8327 0.1203 0.1019 0.1103
5  BASE_X61 in ['建筑业', '制造业']   4018000.0000 0.1240   3123000.0000 0.1235   895000.0000 0.1256 0.2227 1.0128  0.0018  ...   544000.0000 0.1361 0.1354 1.0978  0.0138  0.1116 0.7889 0.1338 0.1379 0.1359
6                        剩余样本  28391000.0000 0.8760  22158000.0000 0.8765  6233000.0000 0.8744 0.2195 0.9982 -0.0128  ...  3453000.0000 0.8639 0.1216 0.9862 -0.0978 -0.1116 0.2111 0.1184 0.8621 0.2082
7  BASINF_X19 in ['硕士', '博士']    140000.0000 0.0049    103000.0000 0.0046    37000.0000 0.0059 0.2643 1.2038  0.0010  ...    22000.0000 0.0064 0.1571 1.2920  0.0014  0.2935 0.8787 0.1923 0.0077 0.0148
8                        剩余样本  28251000.0000 0.9951  22055000.0000 0.9954  6196000.0000 0.9941 0.2193 0.9990 -0.2038  ...  3431000.0000 0.9936 0.1214 0.9986 -0.2920 -0.2935 0.1213 0.1180 0.9923 0.2110
9                        所有规则 718778000.0000 0.9622 655149000.0000 0.9674 63629000.0000 0.9113 0.0885 0.9471 -1.3464  ... 62596000.0000 0.9480 0.0871 0.9853 -0.3741 -0.3888 0.1279 0.0892 0.9364 0.1628

[10 rows x 75 columns]

## 3. 规则集分类器与模型报告

本节重新定义规则、拆分客户并训练分类器，可以独立生成报告。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
from hscredit import Rule, RuleSet, RulesClassifier
from hscredit.core.model_selection import group_train_test_split
from hscredit.report import ModelReport

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

high_query = Rule("BASE_X187 >= 60", name="多头查询偏多", n_jobs=1)
high_accounts = Rule("DK_X1689 >= 12", name="现行账户偏多", n_jobs=1)
industry = Rule("BASE_X61 in ['建筑业', '制造业']", name="指定行业", n_jobs=1)
degree = Rule("BASINF_X19 in ['硕士', '博士']", name="指定学位", n_jobs=1)

features = ["BASE_X187", "DK_X1689"]
target = "MOB6@30"
data[target] = data["MOB6"].gt(30).astype(int)
train, test = group_train_test_split(data, group="用户编码", test_size=0.25, random_state=42)
rule_set = RuleSet(name="多头与账户规则", logic="or", rules=[high_query, high_accounts], n_jobs=1)
model = RulesClassifier(rules=rule_set, target=target, n_jobs=1)
model.fit(train[features + [target]])
display(pd.DataFrame({"实际标签": test[target], "规则分类": model.predict(test[features])}).head())
display(
    pd.DataFrame(model.predict_proba(test[features]), columns=["好样本概率", "坏样本概率"]).head()
)

report = ModelReport(
    model,
    datasets={"训练集": train, "测试集": test},
    features=features,
    overdue=["MOB1", "MOB3", "MOB6"],
    dpds=30,
    overdue_operator=">",
    metrics=["auc", "ks", "accuracy"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_rules_classifier_"))
report.to_excel(
    output / "规则分类器报告.xlsx",
    amount="loan_amt",
    with_plots=False,
    include_sample_records=False,
    mode="strict",
)
print(output)

,实际标签,规则分类
3,0,1
8,0,1
23,0,0
24,0,1
35,0,1


,好样本概率,坏样本概率
0,0.0000,1.0000
1,0.0000,1.0000
2,1.0000,0.0000
3,0.0000,1.0000
4,0.0000,1.0000


统计指标        KS           AUC               样本数              坏样本率       
数据集        训练集    测试集    训练集    测试集        训练集        测试集    训练集    测试集
逾期指标                                                                   
MOB1@30 0.0707 0.0330 0.4647 0.4835 83942.0000 27987.0000 0.0134 0.0139
MOB3@30 0.0304 0.0237 0.4848 0.4882 83942.0000 27987.0000 0.0500 0.0508
MOB6@30 0.0220 0.0149 0.4890 0.4926 83942.0000 27987.0000 0.0905 0.0909

C:\Users\18306\AppData\Local\Temp\hscredit_rules_classifier_0m2yvb4a


## 4. 串行与并行规则流程

RuleFlow 展示流程命中、分组汇总与结果比对。回放结果用于演示接口，不声称拥有线上日志。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from hscredit import Rule, RuleFlow

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

high_query = Rule("BASE_X187 >= 60", name="多头查询偏多", n_jobs=1)
high_accounts = Rule("DK_X1689 >= 12", name="现行账户偏多", n_jobs=1)
industry = Rule("BASE_X61 in ['建筑业', '制造业']", name="指定行业", n_jobs=1)
degree = Rule("BASINF_X19 in ['硕士', '博士']", name="指定学位", n_jobs=1)

rules = [high_query, high_accounts, industry, degree]
serial = RuleFlow(rules, mode="serial", n_jobs=1)
parallel = RuleFlow(rules, mode="parallel", n_jobs=1)
display(serial.predict(data).head())
display(parallel.predict(data).head())
display(serial.summary(data))
display(parallel.summary(data))
display(serial.report(data, group="GROUP"))
display(serial.report(data, date="授信时间"))

replayed_hits = parallel.predict(data)
summary, differences = parallel.compare(data, replayed_hits)
display(summary)
display(differences.head())

,多头查询偏多,现行账户偏多,指定行业,指定学位,命中规则序号,命中规则,是否命中,是否通过
0,False,True,<NA>,<NA>,2,现行账户偏多,True,False
1,True,<NA>,<NA>,<NA>,1,多头查询偏多,True,False
2,False,True,<NA>,<NA>,2,现行账户偏多,True,False
3,False,True,<NA>,<NA>,2,现行账户偏多,True,False
4,True,<NA>,<NA>,<NA>,1,多头查询偏多,True,False


,多头查询偏多,现行账户偏多,指定行业,指定学位,命中规则序号,命中规则,是否命中,是否通过
0,False,True,True,False,2|3,现行账户偏多|指定行业,True,False
1,True,True,False,False,1|2,多头查询偏多|现行账户偏多,True,False
2,False,True,False,False,2,现行账户偏多,True,False
3,False,True,True,False,2|3,现行账户偏多|指定行业,True,False
4,True,True,False,False,1|2,多头查询偏多|现行账户偏多,True,False


,统计类型,样本总数,通过样本,命中样本,通过率,命中率
0,整体合计,111929,5464,106465,0.0488,0.9512


,统计类型,样本总数,通过样本,命中样本,通过率,命中率
0,整体合计,111929,5464,106465,0.0488,0.9512


,GROUP,统计类型,规则序号,规则名称,规则表达式,统计范围样本数,当前规则样本数,规则命中,命中率(统计范围),命中率(当前规则),规则通过,通过率(统计范围),通过率(当前规则)
0,优质客群-公积金类,明细,1,多头查询偏多,BASE_X187 >= 60,1796,1796,873,0.4861,0.4861,923,0.5139,0.5139
1,优质客群-公积金类,明细,2,现行账户偏多,DK_X1689 >= 12,1796,923,849,0.4727,0.9198,74,0.0412,0.0802
2,优质客群-公积金类,明细,3,指定行业,"BASE_X61 in ['建筑业', '制造业']",1796,74,8,0.0045,0.1081,66,0.0367,0.8919
3,优质客群-公积金类,明细,4,指定学位,"BASINF_X19 in ['硕士', '博士']",1796,66,0,0.0000,0.0000,66,0.0367,1.0000
4,优质客群-公积金类,分组合计,<NA>,分组合计,,1796,1796,1730,0.9633,0.9633,66,0.0367,0.0367
5,优质客群-房贷类,明细,1,多头查询偏多,BASE_X187 >= 60,10597,10597,4621,0.4361,0.4361,5976,0.5639,0.5639
6,优质客群-房贷类,明细,2,现行账户偏多,DK_X1689 >= 12,10597,5976,5775,0.5450,0.9664,201,0.0190,0.0336
7,优质客群-房贷类,明细,3,指定行业,"BASE_X61 in ['建筑业', '制造业']",10597,201,23,0.0022,0.1144,178,0.0168,0.8856
8,优质客群-房贷类,明细,4,指定学位,"BASINF_X19 in ['硕士', '博士']",10597,178,2,0.0002,0.0112,176,0.0166,0.9888
9,优质客群-房贷类,分组合计,<NA>,分组合计,,10597,10597,10421,0.9834,0.9834,176,0.0166,0.0166


,统计周期,统计类型,规则序号,规则名称,规则表达式,统计范围样本数,当前规则样本数,规则命中,命中率(统计范围),命中率(当前规则),规则通过,通过率(统计范围),通过率(当前规则)
0,2023-12,明细,1,多头查询偏多,BASE_X187 >= 60,647,647,134,0.2071,0.2071,513,0.7929,0.7929
1,2023-12,明细,2,现行账户偏多,DK_X1689 >= 12,647,513,426,0.6584,0.8304,87,0.1345,0.1696
2,2023-12,明细,3,指定行业,"BASE_X61 in ['建筑业', '制造业']",647,87,15,0.0232,0.1724,72,0.1113,0.8276
3,2023-12,明细,4,指定学位,"BASINF_X19 in ['硕士', '博士']",647,72,0,0.0000,0.0000,72,0.1113,1.0000
4,2023-12,分组合计,<NA>,分组合计,,647,647,575,0.8887,0.8887,72,0.1113,0.1113
5,2024-01,明细,1,多头查询偏多,BASE_X187 >= 60,3209,3209,1022,0.3185,0.3185,2187,0.6815,0.6815
6,2024-01,明细,2,现行账户偏多,DK_X1689 >= 12,3209,2187,1829,0.5700,0.8363,358,0.1116,0.1637
7,2024-01,明细,3,指定行业,"BASE_X61 in ['建筑业', '制造业']",3209,358,43,0.0134,0.1201,315,0.0982,0.8799
8,2024-01,明细,4,指定学位,"BASINF_X19 in ['硕士', '博士']",3209,315,2,0.0006,0.0063,313,0.0975,0.9937
9,2024-01,分组合计,<NA>,分组合计,,3209,3209,2896,0.9025,0.9025,313,0.0975,0.0975


,规则序号,规则名称,规则表达式,样本总数,一致样本,差异样本,一致率,差异率
0,1,多头查询偏多,BASE_X187 >= 60,111929,111929,0,1.0000,0.0000
1,2,现行账户偏多,DK_X1689 >= 12,111929,111929,0,1.0000,0.0000
2,3,指定行业,"BASE_X61 in ['建筑业', '制造业']",111929,111929,0,1.0000,0.0000
3,4,指定学位,"BASINF_X19 in ['硕士', '博士']",111929,111929,0,1.0000,0.0000


,用户编码,授信时间,MOB1,MOB6,BASE_X187,loan_amt,GROUP,BASE_X257,BASE_X304,DK_X1995_2,...,是否一致_多头查询偏多,线下_现行账户偏多,线上_现行账户偏多,是否一致_现行账户偏多,线下_指定行业,线上_指定行业,是否一致_指定行业,线下_指定学位,线上_指定学位,是否一致_指定学位


## 5. 策略置换、迭代报告与规则交付

当前数据仅含放款订单。回测使用样本内基准口径，不外推全申请客群通过率。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
from hscredit import Rule
from hscredit.core.rules import RuleArtifact
from hscredit.core.model_selection import group_train_test_split
from hscredit.report import feature_bin_stats, rule_swap_analysis, swap_out_report
from hscredit.excel import ExcelWriter, dataframe2excel

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

high_query = Rule("BASE_X187 >= 60", name="多头查询偏多", n_jobs=1)
high_accounts = Rule("DK_X1689 >= 12", name="现行账户偏多", n_jobs=1)
industry = Rule("BASE_X61 in ['建筑业', '制造业']", name="指定行业", n_jobs=1)
degree = Rule("BASINF_X19 in ['硕士', '博士']", name="指定学位", n_jobs=1)

reference, analysis_data = group_train_test_split(
    data, group="用户编码", test_size=0.25, random_state=42
)
bin_table = feature_bin_stats(
    reference,
    feature="BASE_X187",
    overdue=["MOB1", "MOB3", "MOB6"],
    dpds=30,
    overdue_operator=">",
    method="quantile",
    max_n_bins=5,
    n_jobs=1,
)
swap = rule_swap_analysis(
    analysis_data,
    score="BASE_X187",
    rules_out=[high_accounts],
    rules_in=[high_query],
    reference_data=reference,
    bin_table=bin_table,
    overdue=["MOB1", "MOB3", "MOB6"],
    dpds=30,
    overdue_operator=">",
    amount="loan_amt",
    sample_survival_rate=1.0,
    out_in_uplift=1.0,
    n_jobs=1,
)
output = Path(mkdtemp(prefix="hscredit_rule_swap_"))
writer = ExcelWriter()
for index, (name, table) in enumerate(swap.items()):
    if isinstance(table, pd.DataFrame):
        display(table)
        dataframe2excel(table, writer, sheet_name=f"{index}-{name}"[:31], index=True)
writer.save(output / "策略置换报告.xlsx")

swap_out_report(
    analysis_data,
    rules=[high_accounts],
    features=["BASE_X187", "DK_X1689", "DK_X1223"],
    overdue=["MOB1", "MOB3", "MOB6"],
    dpds=30,
    overdue_operator=">",
    amount="loan_amt",
    date="授信时间",
    methods="quantile",
    n_jobs=1,
    save=str(output / "策略迭代报告.xlsx"),
)
combined = high_query & high_accounts
artifact = RuleArtifact(
    expression=combined.expr, target_spec={"overdue": ["MOB1", "MOB3", "MOB6"], "dpds": [30]}
)
artifact.save(output / "规则.json")
restored = RuleArtifact.load(output / "规则.json")
display(restored.predict(analysis_data).head())
print(output)

分箱详情                                                                           ... MOB6_30+                                                                                             
        规则分类    指标名称             规则详情   行类型  决策样本数 决策样本占比   样本总数   样本占比 阶段前样本数 阶段后样本数  ...   原始坏样本率   调整后坏样本数 调整后坏样本率  LIFT值    坏账改善    风险拒绝比       原始坏样本总额      调整后坏样本总额 原始坏样本率(金额) 调整后坏样本率(金额)
0       全量样本                             状态  27987 1.0000  27987 1.0000  27987  27987  ...   0.0908 2542.1323  0.0908 1.0000  0.0000   0.0000 16366152.5240 16366152.5240     0.0875      0.0875
1      原策略通过                             状态  27944 0.9985  27944 0.9985  27944  27944  ...   0.0909 2539.0000  0.0909 1.0003 -0.0003  -0.0003 16350000.0000 16350000.0000     0.0876      0.0876
2   IN-OUT置出  现行账户偏多   DK_X1689 >= 12  规则明细  26408 0.9436  26408 0.9436  27944   1536  ...   0.0896 2365.0000  0.0896 0.9859  0.0141   0.0149 15430000.0000 15430000.0000     0.0863      0.0863
3   IN-OUT置出      合计                   阶段合计  26408 0.9436  26408 0.9436  27944   1536  ...   0.0896 2365.0000  0.0896 0.9859  0.0141   0.0149 15430000.0000 15430000.0000     0.0863      0.0863
4    IN-IN通过                             状态   1536 0.0549   1536 0.0549   1536   1536  ...   0.1133  174.0000  0.1133 1.2471 -0.2471  -4.5031   920000.0000   920000.0000     0.1150      0.1150
5   OUT-IN置入  多头查询偏多  BASE_X187 >= 60  规则明细     43 0.0015     43 0.0015   1536   1579  ...   0.0728    3.1323  0.0728 0.8019  0.1981 128.9038    16152.5240    16152.5240     0.0721      0.0721
6   OUT-IN置入      合计                   阶段合计     43 0.0015     43 0.0015   1536   1579  ...   0.0728    3.1323  0.0728 0.8019  0.1981 128.9038    16152.5240    16152.5240     0.0721      0.0721
7  total通过样本                             状态   1579 0.0564   1579 0.0564   1579   1579  ...   0.1122  177.1323  0.1122 1.2350 -0.2350  -4.1656   936152.5240   936152.5240     0.1138      0.1138
8   ALL-IN置换                           策略合计   1579 0.0564   1579 0.0564  27944   1579  ...   0.1122  177.1323  0.1122 1.2350 -0.2350  -4.1656   936152.5240   936152.5240     0.1138      0.1138

[9 rows x 75 columns]

,目标标签,指标,变化前,变化后,绝对变化,相对变化
0,MOB1_30+,通过率,99.8464,5.6419,-94.2045,-0.9435
1,MOB1_30+,逾期率,0.0139,0.0218,0.0079,0.5670
2,MOB1_30+,风险上浮系数,1.0000,1.0000,0.0000,0.0000
3,MOB1_30+,样本集幸存比例,1.0000,1.0000,0.0000,0.0000
4,MOB3_30+,通过率,99.8464,5.6419,-94.2045,-0.9435
5,MOB3_30+,逾期率,0.0509,0.0707,0.0198,0.3897
6,MOB3_30+,风险上浮系数,1.0000,1.0000,0.0000,0.0000
7,MOB3_30+,样本集幸存比例,1.0000,1.0000,0.0000,0.0000
8,MOB6_30+,通过率,99.8464,5.6419,-94.2045,-0.9435
9,MOB6_30+,逾期率,0.0909,0.1122,0.0213,0.2346


3     False
8      True
23    False
24    False
35     True
dtype: bool

C:\Users\18306\AppData\Local\Temp\hscredit_rule_swap_z5fvwhhk
